week 2

In [9]:
from google.colab import files
uploaded = files.upload()

Saving sales.csv to sales.csv


In [10]:
import pandas as pd

df = pd.read_csv("sales.csv")

print(df.head())

   sale_id  store_id  product_id  quantity  price   cost   sale_date
0        1         1         101         2  55000  45000  2026-05-01
1        2         1         102         5    500    300  2026-05-01
2        3         2         103         3   1200    800  2026-05-02
3        4         2         104         1   3500   2500  2026-05-02
4        5         3         105         2   7000   5000  2026-05-03


In [11]:
print(df.isnull().sum())

sale_id       0
store_id      0
product_id    0
quantity      0
price         0
cost          0
sale_date     0
dtype: int64


In [12]:
df = df.dropna()


In [13]:
df['revenue'] = df['quantity'] * df['price']

In [14]:
df['profit'] = df['revenue'] - df['cost']

In [15]:
summary = df.groupby('store_id')[['revenue','profit']].sum()

In [16]:
print(summary)

          revenue  profit
store_id                 
1          119500   71700
2            7100    3800
3           15000    9940
4             400     390
5           59800   14000


week 3

In [18]:
!pip install pyspark

In [20]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Retailsales") \
    .getOrCreate()

In [21]:
df = spark.read.csv(
    "sales.csv",
    header=True,
    inferSchema=True
)

In [22]:
df.show()

+-------+--------+----------+--------+-----+-----+----------+
|sale_id|store_id|product_id|quantity|price| cost| sale_date|
+-------+--------+----------+--------+-----+-----+----------+
|      1|       1|       101|       2|55000|45000|2026-05-01|
|      2|       1|       102|       5|  500|  300|2026-05-01|
|      3|       2|       103|       3| 1200|  800|2026-05-02|
|      4|       2|       104|       1| 3500| 2500|2026-05-02|
|      5|       3|       105|       2| 7000| 5000|2026-05-03|
|      6|       3|       106|      10|  100|   60|2026-05-03|
|      7|       4|       107|      20|   20|   10|2026-05-04|
|      8|       5|       101|       1|55000|45000|2026-05-04|
|      9|       5|       103|       4| 1200|  800|2026-05-05|
|     10|       1|       104|       2| 3500| 2500|2026-05-05|
+-------+--------+----------+--------+-----+-----+----------+



In [23]:
df.filter(df.quantity < 5).show()

+-------+--------+----------+--------+-----+-----+----------+
|sale_id|store_id|product_id|quantity|price| cost| sale_date|
+-------+--------+----------+--------+-----+-----+----------+
|      1|       1|       101|       2|55000|45000|2026-05-01|
|      3|       2|       103|       3| 1200|  800|2026-05-02|
|      4|       2|       104|       1| 3500| 2500|2026-05-02|
|      5|       3|       105|       2| 7000| 5000|2026-05-03|
|      8|       5|       101|       1|55000|45000|2026-05-04|
|      9|       5|       103|       4| 1200|  800|2026-05-05|
|     10|       1|       104|       2| 3500| 2500|2026-05-05|
+-------+--------+----------+--------+-----+-----+----------+



In [25]:

from pyspark.sql.functions import avg

df.groupBy("store_id").agg(avg("quantity")).show()

+--------+-------------+
|store_id|avg(quantity)|
+--------+-------------+
|       1|          3.0|
|       3|          6.0|
|       5|          2.5|
|       4|         20.0|
|       2|          2.0|
+--------+-------------+



week 4

In [63]:
uploaded = files.upload()

Saving products.csv to products (6).csv


In [64]:
products_df = df = spark.read.csv(
    "products.csv",
    header=True,
    inferSchema=True
)

In [65]:
products_df.show()

+----------+------------+-----------+-----+
|product_id|product_name|   category|price|
+----------+------------+-----------+-----+
|       101|      Laptop|Electronics|75000|
|       102|  Headphones|Electronics| 3000|
|       103|    Keyboard|Electronics| 1500|
|       104|     Monitor|Electronics|12000|
|       105|Office Chair|  Furniture| 7000|
|       106|        Desk|  Furniture|15000|
|       107|  Smartphone|Electronics|40000|
|       108|    Notebook| Stationery|  100|
|       109|         Pen| Stationery|   20|
|       110|      Tablet|Electronics|30000|
+----------+------------+-----------+-----+



In [67]:
products_df = products_df.withColumnRenamed(
    "price",
    "product_price"
)

In [68]:
products_df.show()

+----------+------------+-----------+-------------+
|product_id|product_name|   category|product_price|
+----------+------------+-----------+-------------+
|       101|      Laptop|Electronics|        75000|
|       102|  Headphones|Electronics|         3000|
|       103|    Keyboard|Electronics|         1500|
|       104|     Monitor|Electronics|        12000|
|       105|Office Chair|  Furniture|         7000|
|       106|        Desk|  Furniture|        15000|
|       107|  Smartphone|Electronics|        40000|
|       108|    Notebook| Stationery|          100|
|       109|         Pen| Stationery|           20|
|       110|      Tablet|Electronics|        30000|
+----------+------------+-----------+-------------+



In [75]:
uploaded = files.upload()

Saving sales.csv to sales (3).csv


In [76]:
sales_df = df = spark.read.csv(
    "sales.csv",
    header=True,
    inferSchema=True
)

In [77]:
final_df = sales_df.join(
    products_df,
    "product_id"
)

In [78]:
final_df.show()

+----------+-------+--------+--------+-----+-----+----------+------------+-----------+-------------+
|product_id|sale_id|store_id|quantity|price| cost| sale_date|product_name|   category|product_price|
+----------+-------+--------+--------+-----+-----+----------+------------+-----------+-------------+
|       101|      1|       1|       2|55000|45000|2026-05-01|      Laptop|Electronics|        75000|
|       102|      2|       1|       5|  500|  300|2026-05-01|  Headphones|Electronics|         3000|
|       103|      3|       2|       3| 1200|  800|2026-05-02|    Keyboard|Electronics|         1500|
|       104|      4|       2|       1| 3500| 2500|2026-05-02|     Monitor|Electronics|        12000|
|       105|      5|       3|       2| 7000| 5000|2026-05-03|Office Chair|  Furniture|         7000|
|       106|      6|       3|      10|  100|   60|2026-05-03|        Desk|  Furniture|        15000|
|       107|      7|       4|      20|   20|   10|2026-05-04|  Smartphone|Electronics|     

In [79]:
from pyspark.sql.functions import col

In [80]:
final_df = final_df.withColumn(
    "revenue",
    col("quantity") * col("price")
)

In [81]:
final_df = final_df.withColumn(
    "profit",
    col("revenue") - (col("quantity") * col("cost"))
)



In [82]:
final_df = final_df.withColumn(
    "profit_margin",
    (col("profit") / col("revenue")) * 100
)


In [83]:
final_df.show()

+----------+-------+--------+--------+-----+-----+----------+------------+-----------+-------------+-------+------+------------------+
|product_id|sale_id|store_id|quantity|price| cost| sale_date|product_name|   category|product_price|revenue|profit|     profit_margin|
+----------+-------+--------+--------+-----+-----+----------+------------+-----------+-------------+-------+------+------------------+
|       101|      1|       1|       2|55000|45000|2026-05-01|      Laptop|Electronics|        75000| 110000| 20000|18.181818181818183|
|       102|      2|       1|       5|  500|  300|2026-05-01|  Headphones|Electronics|         3000|   2500|  1000|              40.0|
|       103|      3|       2|       3| 1200|  800|2026-05-02|    Keyboard|Electronics|         1500|   3600|  1200| 33.33333333333333|
|       104|      4|       2|       1| 3500| 2500|2026-05-02|     Monitor|Electronics|        12000|   3500|  1000| 28.57142857142857|
|       105|      5|       3|       2| 7000| 5000|2026-

In [84]:
final_df.write.csv(
    "/FileStore/output/",
    header=True
)

Use a Databricks SQL cell to find top 3 best-selling products

SELECT product_name,
       SUM(quantity) AS total_sales
FROM sales
GROUP BY product_name
ORDER BY total_sales DESC
LIMIT 3;